## 순차 데이터와 순환 신경망 (Sequence & RNN)

### 1. 순차 데이터 (Sequential Data/Sequence)

* 순차 데이터: 원소들이 **순서**를 가지고 나열된 데이터
    - 순서가 바뀌면 의미가 달라짐
    - Ex) 주식 가격, 날씨 기록, 음성, 문장, DNA 염기 서열
    - Ex) 문장 "The dog bit the man." vs "The man bit the dog."

#### 1) 순차 데이터 모델링

* 지금까지 다룬 지도 학습은 각 샘플이 서로 **독립**이라고 가정 (독립 동일 분포, IID)
    - Ex) 타이타닉 표, MNIST 이미지: 샘플 순서를 섞어도(shuffle) 학습에 문제 없음
* 순차 데이터는 **이전 값이 다음 값에 영향**을 주므로 이 가정이 성립하지 않음
    - Ex) 오늘 주가는 어제 주가와 무관하지 않음, 오늘 날씨는 어제 날씨와 무관하지 않음
    - 순서 정보를 활용할 수 있는 모델이 필요

#### 2) 순차 데이터 vs 시계열 데이터

* 시계열(time series) 데이터는 순차 데이터의 한 종류
    - 주식 가격, 녹화된 음성 등
* 모든 순차 데이터가 시간 차원을 가지는 것은 아님
    - 텍스트 데이터, DNA 염기 서열

In [ ]:
from IPython.display import Image

# 시계열 데이터 예시
Image(url='https://raw.githubusercontent.com/rickiepark/ml-with-pytorch/main/ch15/figures/15_01.png', width=500)

#### 3) 시퀀스 모델링의 종류

In [ ]:
Image(url='https://raw.githubusercontent.com/rickiepark/ml-with-pytorch/main/ch15/figures/15_02.png', width=500)

* 다대일(many-to-one)
    - 입력: 시퀀스, 출력: 고정크기의 텐서
    - Ex) 감성분석: 입력 - 텍스트, 출력 - 클래스 레이블 (좋음/싫음)
* 일대다(one-to-may)
    - 입력: 고정크기의 텐서, 출력: 시퀀스
    - Ex) 이미지 캡셔닝: 입력 - 이미지, 출력 - 이미지 요약 텍스트
* 다대다(many-to-many)
    - 입력: 시퀀스, 출력: 시퀀스
    - 동기적, 비동기적으로 나눌 수 있음
    - Ex) 품사 태깅(동기적): 입력 - 문장, 출력 - 각 단어의 품사
    - Ex) 번역(비동기적, 지연 존재): 입력 - 문장(A 언어), 출력 - 문장(B 언어)

### 2. 순환 신경망 (Recurrent Neural Network, RNN)

#### 1) 순환 신경망의 구조

In [ ]:
from IPython.display import Image
Image(url='https://wikidocs.net/images/page/22886/rnn_image2_ver3.PNG', width=600)

* 셀(Cell): RNN의 은닉층에서 활성화 함수를 통해 결과를 내보내는 역할을 하는 노드
    - 이전의 값을 기억하려고 하는 일종의 메모리
    - 메모리 셀, RNN 셀
* 은닉 상태(hidden state): 메모리 셀이 출력층 방향으로 또는 다음 시점 $t+1$의 자신에게 보내는 값

In [ ]:
Image(url='https://wikidocs.net/images/page/22886/rnn_image4_ver2.PNG', width=300)

**은닉층**
$$
h_t = tanh(W_x x_t + W_h h_{t-1} +b)
$$

**출력층**
$$
y_t = f(W_y h_t +b)
$$

**왜 은닉층에 tanh를 쓸까?**

* 은닉 상태 $h_t$는 "지금까지 본 것을 기억한 값"이고, 다음 시점으로 넘어가면서 **같은 가중치 $W_h$가 계속 곱해짐**
    - $h_1 \to h_2 \to \cdots \to h_{10}$ 이면 $W_h$가 10번 곱해지는 셈
    - 값이 조금만 커져도 시점을 거듭하며 눈덩이처럼 불어날 수 있음 → 출력 범위를 **일정하게 가두는** 함수가 필요
    - sigmoid와 tanh 모두 출력 범위가 정해져 있어 후보가 됨
* sigmoid가 아닌 tanh를 쓰는 이유
    - sigmoid: 출력이 $(0, 1)$ → 항상 양수, 평균이 0.5 근처
        - "확률"을 표현하는 데는 좋지만, 기억을 담는 값으로 쓰면 한 방향으로만 쌓이고 치우침
    - tanh: 출력이 $(-1, 1)$ → **0을 중심으로 대칭**
        - 증가/감소, 긍정/부정 같은 **양방향 정보**를 표현할 수 있음
        - 0 근처에서 기울기가 1 (sigmoid는 최대 0.25) → 학습 시 기울기가 덜 사라짐
* softmax를 쓰지 않는 이유
    - 모든 원소의 합이 1이 되어야 함 → 한 값이 커지면 다른 값은 작아져야 함
    - 은닉 상태의 각 원소는 서로 다른 정보를 독립적으로 기억해야 하므로 부적합
* 정리: **sigmoid, softmax는 출력층**에서 확률을 만들 때 쓰고, **tanh는 은닉층**에서 기억을 안정적으로 유지할 때 사용
    - 출력층의 $f$는 문제에 따라 달라짐: 이진 분류 → sigmoid, 다중 분류 → softmax, 회귀 → 없음

In [ ]:
Image(url='https://wikidocs.net/images/page/22886/rnn_images4-5.PNG', width=800)

In [ ]:
import torch
import torch.nn as nn

In [ ]:
input_size = 5 # 입력의 크기
hidden_size = 8 # 은닉 상태의 크기

In [ ]:
# (batch_size, time_steps, input_size)
inputs = torch.Tensor(1, 10, input_size)

In [ ]:
cell = nn.RNN(input_size, hidden_size, batch_first=True)

* ```batch_first```: 입력 텐서의 첫번째 차원이 무엇인지 알려줌
    - ```batch_first=False```(default): ```(seq_len, batch, feature)```
    - ```batch_first=True```: ```(batch, seq_len, feature)```


In [ ]:
outputs, _status = cell(inputs)

In [ ]:
print(outputs.shape) # 모든 time-step의 hidden_state

In [ ]:
print(_status.shape) # 최종 time-step의 hidden_state

**`outputs`와 `_status`의 관계**

* `outputs`: **마지막 층**이 **모든 시점**에서 낸 은닉 상태 → `(배치 크기, 시퀀스 길이, 은닉 상태의 크기)`
* `_status`: **모든 층**이 **마지막 시점**에서 낸 은닉 상태 → `(층의 개수, 배치 크기, 은닉 상태의 크기)`

```
                t=1   t=2  ...  t=10
        층 2  →   ●     ●    ...   ◉    ← outputs = 이 행 전체 (마지막 층, 모든 시점)
        층 1  →   ●     ●    ...   ◉
                                ↑
                        _status = 이 열 전체 (모든 층, 마지막 시점)
```

* 두 값이 겹치는 칸(마지막 층의 마지막 시점)이 있으므로 `outputs[:, -1, :]`와 `_status[-1]`은 같은 값

In [ ]:
_status

In [ ]:
print(outputs[:, -1, :])
print(_status[-1]) # 마지막 시점의 출력 == 최종 은닉 상태

**시퀀스 모델링의 종류를 코드로 확인하기**

RNN은 항상 모든 시점의 출력(`outputs`)을 내고, 그중 **무엇을 사용하느냐**에 따라 모델링의 종류가 달라짐

In [ ]:
# 다대일 (many-to-one): 마지막 시점의 출력만 사용 → Ex) 감성분석
last = outputs[:, -1, :]   # (배치 크기, 은닉 상태의 크기)
fc = nn.Linear(8, 2)       # 클래스 2개 (좋음/싫음)
print(fc(last).shape)      # 문장 하나에 답 하나

In [ ]:
# 다대다 (many-to-many, 동기적): 모든 시점의 출력을 사용 → Ex) 품사 태깅
fc = nn.Linear(8, 3)       # 품사 3종류라고 가정
print(fc(outputs).shape)   # 단어 10개 각각에 답 하나씩

* 일대다, 다대다(비동기적)는 출력을 다시 입력으로 넣는 반복 구조나 인코더-디코더 구조가 필요

#### 2) 깊은 순환 신경망(Deep Recurrent Neural Network)

In [ ]:
Image(url='https://wikidocs.net/images/page/22886/rnn_image4.5_finalPNG.PNG', width=600)

In [ ]:
# (batch_size, time_steps, input_size)
inputs = torch.Tensor(1, 10, 5)

In [ ]:
cell = nn.RNN(input_size = 5, hidden_size = 8, num_layers = 2, batch_first=True)
outputs, _status = cell(inputs)

In [ ]:
print(outputs.shape) # 모든 time-step의 hidden_state

In [ ]:
print(_status.shape) # (층의 개수, 배치 크기, 은닉 상태의 크기)

* `_status[-1]`: 마지막 층(2층)의 최종 은닉 상태 → `outputs[:, -1, :]`와 같음
* `_status[0]`: 첫번째 층(1층)의 최종 은닉 상태 → `outputs`에는 포함되지 않음

In [ ]:
print(torch.allclose(outputs[:, -1, :], _status[-1])) # 마지막 층
print(torch.allclose(outputs[:, -1, :], _status[0]))  # 첫번째 층

#### 3) 양방향 순환 신경망(Bidirectional Recurrent Neural Network)

시점 $t$에서의 출력값을 예측할 때 이전 시점의 데이터뿐만 아니라, 이후 데이터로도 예측할 수 있다는 아이디어에 기반

```python
Exercise is very effective at [          ] belly fat.

1) reducing
2) increasing
3) multiplying
```

In [ ]:
Image(url='https://wikidocs.net/images/page/22886/rnn_image5_ver2.PNG', width=600)

* 첫번째 메모리셀: 앞 시점의 은닉 상태 (Forward States)를 전달받음
* 두번째 메모리셀: 뒤 시점의 은닉 상태 (Backward States)를 전달받음

In [ ]:
# 깊은 양방향 순환 신경망
Image(url='https://wikidocs.net/images/page/22886/rnn_image6_ver3.PNG', width=600)

In [ ]:
# (batch_size, time_steps, input_size)
inputs = torch.Tensor(1, 10, 5)

In [ ]:
cell = nn.RNN(input_size = 5, hidden_size = 8, num_layers = 2, batch_first=True, bidirectional = True)

In [ ]:
outputs, _status = cell(inputs)

In [ ]:
print(outputs.shape) # (배치 크기, 시퀀스 길이, 은닉 상태의 크기 x 2)

In [ ]:
print(_status.shape) # (층의 개수 x 2, 배치 크기, 은닉 상태의 크기)

**양방향일 때 `outputs`와 `_status`의 관계**

* `outputs`의 마지막 차원(16)은 **[정방향 8 | 역방향 8]** 을 이어 붙인 것
* 정방향은 $t=1 \rightarrow t=10$ 으로 읽으므로 **마지막 시점**에서 끝남
* 역방향은 $t=10 \rightarrow t=1$ 로 읽으므로 **첫 시점**에서 끝남

In [ ]:
print(torch.allclose(outputs[:, -1, :8], _status[-2])) # 정방향의 최종값: 마지막 시점
print(torch.allclose(outputs[:, 0, 8:], _status[-1]))  # 역방향의 최종값: 첫 시점
print(torch.allclose(outputs[:, -1, 8:], _status[-1])) # 역방향은 마지막 시점이 최종값이 아님

* 양방향 RNN으로 다대일 분류를 할 때 `outputs[:, -1, :]`을 사용하면, 역방향 정보는 단어를 하나만 읽은 상태가 됨
* 정방향과 역방향의 최종 은닉 상태를 이어 붙여서 사용

In [ ]:
last = torch.cat([_status[-2], _status[-1]], dim=1) # (배치 크기, 은닉 상태의 크기 x 2)
print(last.shape)

### 3. LSTM과 GRU

#### 1) 바닐라 RNN의 한계

* 바닐라 RNN(Vanilla RNN): 가장 단순한 형태의 RNN
* 장기의존성 문제 (the problem of Long-Term Dependencies): time step이 길어질 수록 이전 정보가 잘 전달되지 않는 문제

In [ ]:
from IPython.display import Image
Image(url="https://wikidocs.net/images/page/22888/lstm_image1_ver2.PNG", width=400)

Ex. "모스크바에 여행을 왔는데 건물도 예쁘고 먹을 것도 맛있었어. 그런데 글쎄 직장 상사한테 전화가 왔어. 어디냐고 묻더라구 그래서 나는 말했지. 저 여행왔는데요. 여기 ___"

#### 2) LSTM (Long Short-Term Memory)

* 입력 게이트: 장기 기억에 저장될 정보를 결정
* 삭제 게이트: 장기 기억에서 어떤 정보를 유지하거나 폐기할지 결정
* 출력 게이트: 현재 입력, 이전 단기 메모리 및 새로 계산된 장기 메모리를 사용하여 다음 시간 단계에서 셀로 전달될 새로운 단기 메모리를 생성

**사용법**
```python
nn.LSTM(input_dim, hidden_size, batch_first=True)
```

In [ ]:
lstm = nn.LSTM(input_size = 5, hidden_size = 8, batch_first=True)
outputs, (h, c) = lstm(inputs) # 은닉 상태가 (h, c) 튜플로 나옴

print(outputs.shape) # 모든 time-step의 hidden_state
print(h.shape)       # 은닉 상태 (단기 기억)
print(c.shape)       # 셀 상태 (장기 기억)

* `outputs`의 크기는 RNN과 같음
* LSTM은 은닉 상태(`h`)와 함께 **셀 상태**(`c`, cell state)를 반환
    - `outputs, _status = lstm(inputs)`로 받으면 `_status`가 튜플이 되어 `_status.shape`에서 에러 발생

#### 3) GRU (Gated Recurrent Unit)

* 업데이트 게이트: 전달해야 하는 이전 정보의 양을 결정
* 리셋 게이트: 모델에서 무시해야 할 과거 정보의 양을 결정

**사용법**
```python
nn.GRU(input_dim, hidden_size, batch_first=True)
```

In [ ]:
gru = nn.GRU(input_size = 5, hidden_size = 8, batch_first=True)
outputs, _status = gru(inputs) # RNN과 사용법이 같음

print(outputs.shape) # 모든 time-step의 hidden_state
print(_status.shape) # 최종 time-step의 hidden_state

* GRU는 셀 상태 없이 게이트만으로 기억을 조절하므로 RNN과 사용법이 같음

**파라미터 수 비교**

In [ ]:
rnn = nn.RNN(input_size = 5, hidden_size = 8, batch_first=True)

for model in [rnn, gru, lstm]:
    print(type(model).__name__, sum(p.numel() for p in model.parameters()))

* RNN: 가중치 세트 1개
* GRU: 게이트 2개 + 후보 은닉 상태 → RNN의 3배
* LSTM: 게이트 3개 + 후보 셀 상태 → RNN의 4배